# 12. Three-phase houses with a three-phase inverter (3L3P): Volt-VAr validation

**Question.** Does the blind Volt-VAr slope test work for a three-phase inverter when it is applied to the total of the
three house phases of the net meter?

**What the standard says (AS/NZS 4777.2:2020, checked on the page).**
- Clause 3.3.2.1 (printed p. 20): "For three-phase inverters, the inverter shall use the average of the three
  single-phase voltages as the reference voltage to determine the corresponding volt response action."
- Clause 2.11 (printed p. 13): the output current of a three-phase inverter shall be three-phase balanced; inverters
  that can run in a voltage balance mode (Clause 3.3.2.4) may generate unbalanced currents.

**Test (same settings as the single-phase method).**

| | Net meter | Reference |
|---|---|---|
| P, Q per timestamp | sum over the three house phases of `base − net` (each phase with its own monthly 00-04 h baseline) | sum over the three PV circuits |
| Voltage | mean of the three phase voltages (max as a sensitivity) | mean of the three PV-circuit voltages |
| Gate | total P_est ≥ 0.20 × rating (signed) | \|total P\| ≥ 0.20 × rating |

Only timestamps with all three circuits (P, Q and a voltage within 150-300 V) are used. Window 242 ≤ V < 253 V,
q = 100 Q_tot / rating, OLS, assessable if n ≥ 20 and ΔV ≥ 5 V, conformant if the slope is in [−4.000, −2.667] %/V.
A three-phase inverter has no "phase" to choose, so the only blind step is the configuration.

**Populations** (3L3P, reference configuration 3P): A = clean (`use_in_12`), A3 = A with `n_decided` ≥ 3,
B = every reference-3P site (end-to-end), C = excluded in notebook 10 only for blind reasons (selection bias).
CATCH Power sites are left out of the main results (their active power is the uncorrected raw reading) and shown as a
sensitivity.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_vvar_validation as Valid
from bms_sa_review.ami_analysis.lib import ami_phase1p_vvar as P1
from bms_sa_review.ami_analysis.lib import ami_phase3p_vvar as P3
from bms_sa_review.ami_analysis.lib import ami_phase3p_plots as P3Plots

## 1. Settings

In [ ]:
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]

# Method settings: identical to notebooks 07 and 11
NIGHT_HOURS = (0, 4)
P_MIN_FRAC = Fleet.P_MIN_FRAC
V_LO, V_HI = 242, 253
V_PLAUSIBLE = (150, 300)              # readings outside are dropped before averaging
MIN_N_WINDOW, MIN_V_SPREAD = 20, 5.0
SLOPE_TOLERANCE_PCT = 20              # band [-4.000, -2.667] %/V
FIT_KW = dict(night_hours=NIGHT_HOURS, p_min_frac=P_MIN_FRAC, v_lo=V_LO, v_hi=V_HI, v_plausible=V_PLAUSIBLE)

# Notebook 12 settings
EXCLUDE_CATCH = True                  # CATCH Power sites out of the main results (sensitivity in section 9)
PHASE_GATE_FRAC = P_MIN_FRAC / 3      # per-phase fits: each phase carries about 1/3 of the output
MIN_TOTAL_SLOPE = -1.0                # per-phase shares only where the total slope is at least this steep (%/V)
SHARE_FLAG = 0.2                      # flag a site-month if a PV circuit's share of the total slope is below this
PV_OFF_FRAC, DAY_START, DAY_END = 0.05, 7, 18   # phantom-export check (as notebook 11)
N_BOOT, SEED = 1000, 42
SAMPLE_N_SITES = None                 # e.g. 40 for a quick test run (seeded random sites); None = all sites

DUCKDB_MEMORY, DUCKDB_THREADS = "8GB", 4
OVERWRITE = False

PHASE_DIR = Fleet.store_dir() / "vvar_phase" / "phase_config"      # notebook 10
DIR_3L1P = Fleet.store_dir() / "vvar_phase" / "3l1p_vvar"           # notebook 11
FLEET_DIR = Fleet.store_dir() / "vvar_fleet" / f"2024_2025_band_{V_LO}_{V_HI}"   # notebook 07 (1L1P)
OUT_DIR = Fleet.store_dir() / "vvar_phase" / "3l3p_vvar"
OUT_DIR.mkdir(parents=True, exist_ok=True)
KEYS = P3.KEYS
in_months = lambda d: d[(d.year.astype(str) + "-" + d.month.map("{:02d}".format)).isin(MONTHS)]
print(f"Outputs -> {OUT_DIR}")

## 2. Populations

From notebook 10 (`site_evidence`, `blind_config_site_month`) and the circuit metadata (CATCH Power flag).
No phase-reference check applies to a three-phase inverter, so C is every site excluded only for blind reasons.

In [ ]:
ev = pd.read_parquet(PHASE_DIR / "site_evidence.parquet")
blind_month = pd.read_parquet(PHASE_DIR / "blind_config_site_month.parquet")
circuit_meta = pd.read_parquet(Fleet.Config.store_path("ami_circuit_metadata"))

pop = P3.populations(ev, circuit_meta)
if SAMPLE_N_SITES:
    pop = pop.sample(n=min(SAMPLE_N_SITES, len(pop)), random_state=SEED).reset_index(drop=True)
    print(f"TEST RUN: {len(pop)} random sites")
sites = pop[["site_id", "rating_kw"]]

display(pd.DataFrame({"sites": pop[["in_B", "in_A", "in_A3", "in_C"]].sum(),
                      "of which CATCH Power": pop[pop.catch][["in_B", "in_A", "in_A3", "in_C"]].sum()}))
display(pop.exclusion_reason.value_counts().rename("sites").to_frame())
ok = blind_month[blind_month.month_ok & blind_month.site_id.isin(pop[pop.in_A].site_id)]
print("month_ok site-months of A, by year:", ok.groupby("year").size().to_dict(), "| total", len(ok))

## 3. Data checks

1. **Coverage.** Good months per clean site, and clean sites with a good month in each calendar month, against 3L1P.
2. **Month mix.** Share of good site-months in each calendar month, 3L3P vs 3L1P.
3. **CATCH Power.** The Phase-5 build (`synthetic_ami_creation/notebooks/05_ami_build.ipynb`) used
   `apply_power_correction=False`, so these circuits' active power is the raw reading that Phase 3 found unreliable
   (reactive power is always derived from the energy register). They are left out of the main results.

In [ ]:
okm = blind_month[blind_month.month_ok & (blind_month.exclusion_reason == "clean")]
per_site = okm.groupby(["expected", "site_id"]).size().rename("good_months").reset_index()
display(per_site.groupby("expected").good_months.describe().round(1)
        .rename(index={"1P": "3L1P (notebook 11)", "3P": "3L3P"}))

mix = (100 * pd.crosstab(okm.month, okm.expected, normalize="columns")).round(1)
mix.columns = [f"{c} month_ok %" for c in mix.columns]
mix["3L3P sites with a good month"] = okm[okm.expected == "3P"].groupby("month").site_id.nunique()
display(mix)
two = okm[(okm.expected == "3P") & okm.site_id.isin(ev[ev.n_decided == 2].site_id)]
print(f"3P sites with n_decided = 2: {two.site_id.nunique()} sites, {len(two)} good months, "
      f"{(two.month.isin([1, 2])).sum()} of them in January-February")

c = pop[pop.catch]
print(f"CATCH Power: {len(c)} reference-3P sites, {c.in_A.sum()} clean, "
      f"{ok[ok.site_id.isin(c.site_id)].shape[0]} good site-months")

## 4. Fits (cached per month)

- **Totals** (`P3.total_fit_sql`): one query per month and source gives the fit on the average voltage and on the
  maximum phase voltage.
- **Per phase** (`Fleet.run_months`, unchanged): one fit per house phase and per PV circuit, with the gate at
  `PHASE_GATE_FRAC` × rating, for the Clause 2.11 balance check and the household load imbalance.
- **Reference check.** PV-circuit P is positive for generation. A site-month where some, but not all, PV circuits
  have a negative midday median P has a reversed PV circuit, so its summed P and Q are wrong (found while testing:
  site 153366397, January 2025, where the PV total was about a third of the net-meter total). These site-months are
  marked `pv_polarity_mixed` and not used as a reference.

In [ ]:
P1.check_site_list(OUT_DIR, sites, OVERWRITE)
for name, source in [("total_meter", "meter_baseline"), ("total_truth", "ground_truth")]:
    P3.run_total_months(MONTHS, OUT_DIR / name, sites, source=source, overwrite=OVERWRITE,
                        memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, **FIT_KW)
for name, source in [("phase_meter", "meter_baseline"), ("phase_truth", "ground_truth")]:
    Fleet.run_months(MONTHS, OUT_DIR / name, sites, source=source, overwrite=OVERWRITE,
                     memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS,
                     night_hours=NIGHT_HOURS, p_min_frac=PHASE_GATE_FRAC, v_lo=V_LO, v_hi=V_HI)

status_kw = dict(min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD, tolerance_pct=SLOPE_TOLERANCE_PCT,
                 v_lo=V_LO, v_hi=V_HI)
total_meter = P3.add_fit_status(in_months(P3.load_total(OUT_DIR / "total_meter")), **status_kw)
total_truth = P3.add_fit_status(in_months(P3.load_total(OUT_DIR / "total_truth")), **status_kw)
phase_meter = in_months(Fleet.load_results(OUT_DIR / "phase_meter"))
phase_truth = in_months(Fleet.load_results(OUT_DIR / "phase_truth"))
total_meter, total_truth = [d[d.site_id.isin(sites.site_id)] for d in (total_meter, total_truth)]

for name, d in [("net meter", total_meter), ("PV circuits", total_truth)]:
    a = d[d.v_mode == "avg"]
    print(f"{name}: {a.n_complete.sum() / max(a.n_ts.sum(), 1):.1%} of timestamps have all three circuits")
    display(pd.crosstab(a.year, a.fit_status, margins=True))

## 5. Oracle validation (population A, `month_ok` months)

Net-meter total vs PV-circuit total, on site-months where both are assessable (positive class = conformant).
Compared with 1L1P (notebook 06) and 3L1P (notebook 11), overall and split into January-February vs the other months,
because the 3L3P sample is summer-heavy.

In [ ]:
orc_all = P3.oracle_table(total_meter, total_truth, pop, blind_month, v_mode="avg")
orc = orc_all[~orc_all.catch] if EXCLUDE_CATCH else orc_all
A = orc[orc.in_A & orc.month_ok]
main = A[A.both_ok].copy()
main["model_hint"] = main.model_hint.fillna("none")
main["season"] = P3.summer_group(main)

print(f"A, month_ok site-months with a reference row: {len(A):,}")
display(pd.DataFrame({"net meter (total)": A.fit_status_meter.value_counts(),
                      "PV circuits (total)": A.fit_status_truth.value_counts()}).fillna(0).astype(int))
print(f"Both assessable: {len(main):,} site-months at {main.site_id.nunique():,} sites")

cols = ["sites", "n", "TP", "FP", "FN", "TN", "accuracy", "precision", "recall", "specificity", "f1",
        "pass_rate_tested_%", "pass_rate_reference_%"]
oracle_metrics = pd.concat([P1.metrics(main), P1.metrics(main, "year"), P1.metrics(main, "model_hint"),
                            P1.metrics(main, "season")], keys=["all", "year", "model hint", "season"])
display(oracle_metrics[cols].round(3))

In [ ]:
# Like-for-like by season: 1L1P (notebook 07 results), 3L1P (notebook 11 outputs), 3L3P (here)
one = P3.load_1l1p_main(in_months(pd.read_parquet(FLEET_DIR / "site_month_results.parquet")), SLOPE_TOLERANCE_PCT)
three1 = P3.load_3l1p_main(DIR_3L1P)
rows = {}
for name, d in [("1L1P", one), ("3L1P", three1), ("3L3P", main)]:
    d = d.assign(season=P3.summer_group(d))
    for grp, g in [("all", d), *d.groupby("season")]:
        m = P1.metrics(g).iloc[0]
        rows[(name, grp)] = m[["sites", "n", "accuracy", "f1", "pass_rate_reference_%"]]
cohort_compare = pd.DataFrame(rows).T
display(cohort_compare.round(3))

In [ ]:
# Accuracy on inverters that respond (PV-circuit slope <= RESPONSE_SLOPE) vs little or no response.
# Most site-months are inverters with little or no Volt-VAr response, which are easy to classify, so the headline
# accuracy depends on the mix. The "responds" rows show how well the test does where it matters.
RESPONSE_SLOPE = -1.0
rows = {}
for name, d in [("1L1P", one), ("3L1P", three1), ("3L3P", main)]:
    d = d.assign(response=np.where(d.slope_truth <= RESPONSE_SLOPE, "responds (PV slope <= -1 %/V)",
                                   "little or no response"))
    for grp, g in d.groupby("response"):
        rows[(name, grp)] = P1.metrics(g).iloc[0][["sites", "n", "accuracy", "recall", "precision",
                                                   "pass_rate_reference_%"]]
response_compare = pd.DataFrame(rows).T
display(response_compare.round(3))
response_compare.to_csv(OUT_DIR / "response_compare.csv")

In [ ]:
fig = Valid.plot_confusion_matrices(main, pairs=[("blind", "truth")])
fig.axes[0].set_ylabel("Net meter, total of three phases", fontsize=9)
fig.axes[0].set_xlabel("PV circuits, total (reference)", fontsize=9)
fig.savefig(OUT_DIR / "oracle_confusion.png", bbox_inches="tight")
plt.show()
Fleet.plot_blind_vs_truth(main)
plt.show()

## Daily Plots

In [ ]:
from bms_sa_review.ami_analysis.lib import interactive_plots as IPlots

cache = IPlots.explore_3p(main, ev=ev)        # outcome, order, case and day are chosen in the widget

## 6. Main result vs the stricter subset

Same rule as notebook 11: if the A3 accuracy and F1 both fall inside A's site-bootstrap 95% intervals, the 2-month
result is the main one.

In [ ]:
ci = pd.DataFrame({
    "main (n_decided >= 2)": P1.bootstrap_ci(main, N_BOOT, SEED),
    "sensitivity (n_decided >= 3)": P1.bootstrap_ci(main[main.in_A3], N_BOOT, SEED),
}).T
display(ci.round(4))
m, s = ci.iloc[0], ci.iloc[1]
close = (m.accuracy_lo <= s.accuracy <= m.accuracy_hi) and (m.f1_lo <= s.f1 <= m.f1_hi)
print(f"A3 inside the main intervals: {close} -> reported result: "
      f"{'main (n_decided >= 2)' if close else 'sensitivity (n_decided >= 3)'}")
by_year = pd.concat({"main": P1.metrics(main, "year"), "n_decided >= 3": P1.metrics(main[main.in_A3], "year")}, axis=1)
display(by_year.loc[:, (slice(None), ["sites", "n", "accuracy", "f1"])].round(3))

## 7. End-to-end blind route (population B)

Every reference-assessable site-month of every reference-3P site, routed by that month's blind configuration:
`3P` → total test; `1P` → wrong route, **counted as an error**; anything else → not assessable. There is no phase step.

In [ ]:
e2e = P3.route_end_to_end(orc)
display(P3.step_table(e2e))
e2e_summary = P3.end_to_end_summary(e2e)
display(e2e_summary)
print("By year:")
display(pd.concat({y: P3.end_to_end_summary(g)[["site_months", "accuracy"]] for y, g in e2e.groupby("year")}, axis=1))
print("Clean (A) vs other reference-3P sites:")
display(pd.concat({"A": P3.end_to_end_summary(e2e[e2e.in_A])[["site_months", "accuracy"]],
                   "not A": P3.end_to_end_summary(e2e[~e2e.in_A])[["site_months", "accuracy"]]}, axis=1))
routed = e2e[e2e.outcome.isin(["agree", "disagree"])]
print("Given a 3P route (positive = conformant):")
display(P1.metrics(routed)[cols].round(3))

## 8. Selection-bias check

C (excluded only for blind reasons) under a reference-only month filter (both totals assessable, no blind
information), with A under the same filter.

In [ ]:
groups = {
    "A, month_ok (main)": main,
    "A, reference-only month filter": orc[orc.in_A & orc.both_ok],
    "C, reference-only month filter": orc[orc.in_C & orc.both_ok],
}
bias = pd.DataFrame({k: P1.bootstrap_ci(g, N_BOOT, SEED) if len(g) else {} for k, g in groups.items()}).T
display(bias.round(4))
c_only = groups["C, reference-only month filter"]
if len(c_only):
    display(P1.metrics(c_only, "exclusion_reason")[["sites", "n", "accuracy", "f1"]].round(3))

## 9. Sensitivities

1. **Maximum phase voltage** instead of the average (both sources).
2. **Per-phase slopes (Clause 2.11).** Each PV circuit's slope as a share of the total slope (about 1/3 if balanced),
   and the sum of the three against the total. Site-months with a share below `SHARE_FLAG` are flagged, not excluded:
   an inverter in voltage balance mode may legitimately be unbalanced.
3. **Household load imbalance**: the largest house phase's share of the night baseline (1/3 = balanced), in terciles.
4. **CATCH Power** sites on their own.

In [ ]:
# 9.1 Maximum phase voltage
orc_max = P3.oracle_table(total_meter, total_truth, pop, blind_month, v_mode="max")
orc_max = orc_max[~orc_max.catch] if EXCLUDE_CATCH else orc_max
main_max = orc_max[orc_max.in_A & orc_max.month_ok & orc_max.both_ok]
display(pd.DataFrame({"average V (main)": P1.metrics(main).iloc[0], "max phase V": P1.metrics(main_max).iloc[0]})
        .loc[["sites", "n", "accuracy", "f1", "pass_rate_tested_%", "pass_rate_reference_%"]].round(3))
both_modes = main[KEYS + ["pass_truth", "slope_truth"]].merge(
    main_max[KEYS + ["pass_truth", "slope_truth"]], on=KEYS, suffixes=("_avg", "_max"))
print(f"Reference verdict, average vs max voltage: same in "
      f"{(both_modes.pass_truth_avg == both_modes.pass_truth_max).mean():.1%} of {len(both_modes):,} site-months; "
      f"median slope {both_modes.slope_truth_avg.median():.3f} vs {both_modes.slope_truth_max.median():.3f} %/V")

In [ ]:
# 9.2 Per-phase slopes as a share of the total
shares_pv = P3.phase_shares(phase_truth, total_truth, min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD,
                            min_total_slope=MIN_TOTAL_SLOPE)
shares_m = P3.phase_shares(phase_meter, total_meter, min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD,
                           min_total_slope=MIN_TOTAL_SLOPE)
sp = shares_pv.merge(main[KEYS], on=KEYS)
sm = shares_m.merge(main[KEYS], on=KEYS)
display(pd.DataFrame({"PV circuits": sp[["min_share", "max_share", "sum_ratio"]].describe(percentiles=[.1, .5, .9]).T.stack(),
                      "house phases": sm[["min_share", "max_share", "sum_ratio"]].describe(percentiles=[.1, .5, .9]).T.stack()})
        .unstack().round(3))
flag = main.merge(sp[KEYS + ["min_share"]], on=KEYS, how="left")
flag["pv_balance"] = np.select([flag.min_share.isna(), flag.min_share < SHARE_FLAG],
                               ["no per-phase share", f"min share < {SHARE_FLAG}"], default="balanced")
display(P1.metrics(flag, "pv_balance")[["sites", "n", "accuracy", "f1"]].round(3))

**Reading 9.2.** Per-phase shares are only computed where the PV total slope is at least as steep as
`MIN_TOTAL_SLOPE` (−1 %/V) and all three per-phase fits are assessable, i.e. for inverters that clearly respond.
The "no per-phase share" group is therefore mostly inverters with little or no response, which are easy to
classify; its high accuracy says nothing about balance. The comparison that matters is "balanced" vs
"min share < 0.2". The per-phase slopes sum to less than the total (median ratio below 1) because the per-phase gate
(rating / 3) keeps different intervals from the total fit.

In [ ]:
# 9.3 Household load imbalance (night baselines of the three house phases)
imb = P3.load_imbalance(phase_meter)
mi = main.merge(imb, on=KEYS, how="left")
mi["imbalance_tercile"] = pd.qcut(mi.night_max_share, 3, labels=["low", "middle", "high"])
display(mi.groupby("imbalance_tercile", observed=True).night_max_share.describe().round(3))
mt = mi.dropna(subset=["imbalance_tercile"]).astype({"imbalance_tercile": str})
display(P1.metrics(mt, "imbalance_tercile").reindex(["low", "middle", "high"])[["sites", "n", "accuracy", "f1"]].round(3))

In [ ]:
# 9.4 CATCH Power sites
catch_main = orc_all[orc_all.catch & orc_all.in_A & orc_all.month_ok & orc_all.both_ok]
display(pd.DataFrame({"main (no CATCH Power)": P1.metrics(main).iloc[0],
                      "CATCH Power only": P1.metrics(catch_main).iloc[0] if len(catch_main) else pd.Series(dtype=float)})
        .loc[["sites", "n", "accuracy", "f1", "pass_rate_tested_%", "pass_rate_reference_%"]].round(3))

## 10. Error diagnostics

1. **Phantom export** (notebook 11): share of the net-meter total fit's intervals where the PV circuits produce
   almost nothing (|P_tot| < `PV_OFF_FRAC` × rating), and the reference-free version (outside daylight).
2. **Household reactive power on the total**: e = Q_tot,meter − Q_tot,PV (% of rating) on intervals in both fit
   windows, β_e = cov(V, e)/var(V) and sd(e), against 1L1P and 3L1P from notebook 11's caches. The total carries
   the whole house's load, so it is expected to look more like 1L1P than 3L1P.
3. **FN and FP plots.**

In [ ]:
units = main[KEYS + ["rating_kw"]]
diag = P3.run_total_diag(units, MONTHS, OUT_DIR / "diagnostics",
                         f"total_oracle_pv{PV_OFF_FRAC:g}_day{DAY_START}-{DAY_END}", memory_limit=DUCKDB_MEMORY,
                         threads=DUCKDB_THREADS, overwrite=OVERWRITE, day_start=DAY_START, day_end=DAY_END,
                         pv_off_frac=PV_OFF_FRAC, **FIT_KW)
dg = P3Plots.add_outcome(main).merge(diag, on=KEYS)
assert len(dg) == len(main) and (dg.n_window_fit == dg.n_window_meter).all(), "interval counts differ from the fits"
dg["share_pv_off"] = dg.n_pv_off / dg.n_window_fit
dg["share_outside_daylight"] = dg.n_outside_daylight / dg.n_window_fit
phantom_table = P1.outside_daylight_table(dg, "share_pv_off")
display(phantom_table.round(3))
display(P1.outside_daylight_table(dg, "share_outside_daylight").round(3))
dg["phantom_group"] = np.where(dg.share_pv_off > 0.10, "> 10% of fitted intervals with no PV output", "<= 10%")
display(P1.metrics(dg, "phantom_group")[["sites", "n", "TP", "FP", "FN", "TN", "accuracy", "f1"]].round(3))

In [ ]:
# beta_e: 3L3P total vs notebook 11's caches (1L1P sample, 3L1P paired phase)
def read_cache(folder):
    files = sorted(Path(folder).glob("*.parquet"))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()

b1 = read_cache(DIR_3L1P / "beta_e" / "1l1p_sample_n3000_seed42")
b31 = read_cache(DIR_3L1P / "beta_e" / "3l1p_oracle")
b33 = dg[KEYS + ["beta_e", "e_sd", "n_shared", "slope_err"]]
display(P1.distribution_table({"1L1P sample": b1, "3L1P paired phase": b31, "3L3P total": b33},
                              ["beta_e", "e_sd", "n_shared"]).round(3))
print(f"3L3P total: corr(beta_e, slope error) = {b33.beta_e.corr(b33.slope_err):.3f} (n = {b33.beta_e.notna().sum():,})")

In [ ]:
display(P3Plots.outcome_overview(main))
cases = P3Plots.select_cases(main, outcome="FN", n=6, how="random")   # how="worst" / "borderline"; n=None = all
display(cases.round(3))
for _, case in cases.iterrows():
    fig, data = P3Plots.plot_case(case)
    plt.show()

In [ ]:
cases_fp = P3Plots.select_cases(main, outcome="FP", n=6, how="random")
display(cases_fp.round(3))
for _, case in cases_fp.iterrows():
    fig, data = P3Plots.plot_case(case)
    plt.show()

## 11. Coverage gap: proposed Athena check

Most 3L3P sites have one short block of data, and the gap is already in `ami_extract`. The query below (not run
here) counts rows per month in the Solar Analytics `ts` table for a sample of PV circuits of clean sites with few
good months. Run it together with notebook 11's query for the missing 3L1P circuits.

In [ ]:
short = per_site[(per_site.expected == "3P") & (per_site.good_months <= 3)].site_id
sample_circuits = (circuit_meta[circuit_meta.site_id.isin(short) & (circuit_meta.circuit_type == "pv_site_net")]
                   .sort_values("circuit_id").groupby("site_id").head(1).head(20).circuit_id)
print(P1.athena_presence_sql(sample_circuits))

In [ ]:
# 11. Coverage gap: does the Solar Analytics source have more data than the local extract?
from bms_sa_review.synthetic_ami_creation.lib import ami_athena as Athena

N_SAMPLE = 20
short = per_site[(per_site.expected == "3P") & (per_site.good_months <= 3)].site_id
pv_c = circuit_meta[circuit_meta.site_id.isin(short) & (circuit_meta.circuit_type == "pv_site_net")]
sample = (pv_c.sort_values("circuit_id").groupby("site_id").head(1)                 # one PV circuit per site
          .sample(n=min(N_SAMPLE, pv_c.site_id.nunique()), random_state=SEED))      # random, not the lowest ids
circ = sample[["site_id", "circuit_id"]].astype("int64")
print(f"{len(short)} clean 3L3P sites with <= 3 good months; sampled {len(circ)} PV circuits")

# Local: rows per circuit and month in ami_extract (PV files)
parts = []
con = Fleet.connect(DUCKDB_MEMORY, DUCKDB_THREADS)
try:
    con.register("_circ", circ[["circuit_id"]])
    for month in MONTHS:
        if Fleet.month_dir("ami_extract", month).exists():
            parts.append(con.execute(P1.presence_sql("ami_extract", month, "part-pv-*.parquet")).df())
finally:
    con.close()
local = pd.concat(parts, ignore_index=True)
local_months = local.groupby("circuit_id").ym.nunique().rename("months_local")

# Athena: same count in the source table (partition filter on year and is_pv keeps the scan small)
Athena.require_credentials()
sql = P1.athena_presence_sql(circ.circuit_id).replace(
    "WHERE circuit_id IN", "WHERE year IN (2024, 2025) AND is_pv = true\n  AND circuit_id IN")
print(sql)
src = Athena.aq(sql, label="nb12 coverage gap: PV circuits of short 3L3P sites")
display(Athena.scan_report())

src["ym"] = pd.to_datetime(src.month_utc).dt.strftime("%Y-%m")
src_months = src.groupby("circuit_id").ym.nunique().rename("months_athena")
cmp = (circ.set_index("circuit_id").join(local_months).join(src_months).fillna(0).astype(int)
       .assign(gap=lambda d: d.months_athena - d.months_local))
display(cmp.sort_values("gap", ascending=False))
print(f"Circuits with more months in Athena than locally: {(cmp.gap > 0).sum()} of {len(cmp)}; "
      f"median months local {cmp.months_local.median():.0f} vs Athena {cmp.months_athena.median():.0f}")

## 12. Outputs

| File | Content |
|---|---|
| `site_month_results_3l3p.parquet` | `Phase.RESULT_COLUMNS`; `source` = `meter_total` (average V), `meter_total_vmax`, `ground_truth_total` (average V); `unit_id = site_id`; all reference-3P sites, CATCH Power included (flag in the details file) |
| `site_month_details_3l3p.parquet` | per site-month: populations, CATCH flag, `month_ok`, blind configuration, end-to-end outcome, per-phase shares, load imbalance |
| `*.csv` | metric tables |

In [ ]:
results = pd.concat([
    P3.to_result_columns(total_meter[total_meter.v_mode == "avg"], source="meter_total"),
    P3.to_result_columns(total_meter[total_meter.v_mode == "max"], source="meter_total_vmax"),
    P3.to_result_columns(total_truth[total_truth.v_mode == "avg"], source="ground_truth_total"),
], ignore_index=True)
assert not results.duplicated(["site_id", "year", "month", "source"]).any()
results.to_parquet(OUT_DIR / "site_month_results_3l3p.parquet", index=False)

details = (orc_all[KEYS + ["in_A", "in_A3", "in_C", "catch", "month_ok", "blind_config", "exclusion_reason",
                           "model_hint", "n_complete_meter", "n_complete_truth"]]
           .merge(P3.route_end_to_end(orc_all)[KEYS + ["route", "outcome"]], on=KEYS, how="left")
           .merge(shares_pv[KEYS + ["min_share", "max_share", "sum_ratio"]].add_suffix("_pv")
                  .rename(columns={f"{k}_pv": k for k in KEYS}), on=KEYS, how="left")
           .merge(imb, on=KEYS, how="left"))
details.to_parquet(OUT_DIR / "site_month_details_3l3p.parquet", index=False)

oracle_metrics.to_csv(OUT_DIR / "oracle_metrics.csv")
cohort_compare.to_csv(OUT_DIR / "cohort_compare.csv")
ci.to_csv(OUT_DIR / "main_vs_strict.csv")
P3.step_table(e2e).to_csv(OUT_DIR / "end_to_end_steps.csv")
e2e_summary.to_csv(OUT_DIR / "end_to_end_summary.csv")
bias.to_csv(OUT_DIR / "selection_bias.csv")
phantom_table.to_csv(OUT_DIR / "phantom_export.csv")
display(results.groupby("source").fit_status.value_counts().unstack(fill_value=0))
print(f"Saved to {OUT_DIR}")

## 13. Reading the results (full run, 6 Nov 2026)

- **Populations (2).** B 972, A 705, A3 520, C 267 sites; CATCH Power 48 of B (35 clean). A has 3,524 `month_ok`
  site-months (1,798 in 2024, 1,726 in 2025).
- **Data checks (3).**
  - Coverage: a clean 3L3P site has a median of 4 good months, against 13 for 3L1P.
  - Month mix: January-February hold 30.2% of the good months (15.2% + 15.0%). Sites with `n_decided` = 2 contribute
    370 good months, 300 of them in January-February.
- **Fits (4).** All three circuits are present in 99.4% of net-meter and 99.6% of PV-circuit timestamps. 37
  reference site-months have a reversed PV circuit (`pv_polarity_mixed`).
- **Oracle (5).** Both totals assessable in 1,241 site-months at 334 sites (CATCH Power excluded).
  - Accuracy 0.933, F1 0.818 (TP 186, FP 8, FN 75, TN 972); precision 0.959, recall 0.713.
  - By year: 2024 0.922 / 0.738; 2025 0.946 / 0.877.
  - By model hint: 3P 0.970 / 0.850; none 0.903 / 0.807.
  - Single-phase cohorts: 1L1P 0.945 / 0.873; 3L1P 0.970 / 0.908.
  - The summer-heavy mix does not drive the result: January-February 0.937 / 0.806, March-December 0.931 / 0.823.
  - **Inverters that respond.** Where the PV slope is ≤ −1 %/V, accuracy is 0.837 (1L1P), 0.893 (3L1P) and 0.745
    (3L3P), with recall 0.834 / 0.877 / 0.713. Where the inverter barely responds it is 0.994-0.997 in every cohort.
    The headline accuracy therefore depends on how many inverters respond. The three-phase test is weakest where it
    matters: it misses about 29% of conformant inverters, and false positives are rare (8).
- **Main vs stricter (6).** A3: 0.930 / 0.813 on 1,119 site-months, inside A's 95% intervals (accuracy 0.908-0.955,
  F1 0.745-0.872), so the 2-month result is the main one.
- **End-to-end (7).** 1,584 reference-assessable site-months:
  - 126 undecided (8.0%), 16 routed 1P (1.0%), 1,442 routed 3P;
  - accuracy 0.923 at 85.7% coverage; 0.934 given a 3P route (F1 0.814);
  - A sites 0.932, other sites 0.819.
- **Selection bias (8).**
  - C: 144 site-months at 87 sites, accuracy 0.917 (0.862-0.969), F1 0.667 (0.353-0.816).
  - A under the same filter: 0.927 / 0.804.
  - Accuracy is similar. F1 is lower but very uncertain, so there is no clear sign that the clean selection inflates
    the result.
- **Sensitivities (9).**
  - **Maximum phase voltage.** The reference pass rate falls from 21.0% to 11.1%, and the reference verdict is the same
    in 90.9% of 1,226 site-months. This is consistent with inverters responding to the average voltage, as Clause
    3.3.2.1 requires; it is not proof.
  - **Per-phase shares** (276 responding site-months): median smallest PV share 0.227, median largest 0.304.
    Accuracy is 0.710 on "balanced" and 0.817 on "min share < 0.2" site-months, so there is no sign that imbalance
    hurts. Small shares are flagged, not excluded (Clause 2.11 allows imbalance in voltage balance mode).
  - **Household load imbalance.** The metric is fixed to ignore months with a negative night baseline (re-run 9.3 to
    update the table above). After the fix, 1,217 of 1,241 site-months have a share, and accuracy by tercile is 0.924
    / 0.938 / 0.936 (low / middle / high): no effect.
  - **CATCH Power** on its own: 0.863 / 0.688 on 73 site-months, so leaving it out of the main result is justified.
- **Error diagnostics (10).**
  - **Phantom export:** 29 site-months (20 sites) have more than 10% of fitted intervals with no PV output. Accuracy
    there is 0.862, and they hold 4 of the 75 FN, so it is rarer than in 3L1P.
  - **Household reactive power on the total:** median |β_e| 0.162 %/V and median sd(e) 2.041% of rating, against
    0.108 / 1.782 for 1L1P and 0.085 / 1.421 for the single 3L1P phase. Its correlation with the slope error is 0.895.
    The total carries the whole house's load on three phases, which is the most likely reason for the lower recall.
  - **FN plots:** 58 of 75 FN are too shallow (median net-meter slope −1.568 vs PV −3.186). In the plotted cases,
    fast-switching household loads add large swings to the net-meter Q. In the FP example (site 976284004, March
    2025), a load stepping off the night baseline holds P_est near the gate outside PV hours, and the household's own
    reactive power creates a slope the PV circuits do not show.
- **For notebook 13.**
  - Report conformance per cohort before pooling: recall differs (0.834 / 0.877 / 0.713), so a pooled rate understates
    three-phase conformance most. In this validation set the net-meter pass rate is 15.6%, against 21.0% by the
    reference.
  - Leave out CATCH Power.
  - Carry the short coverage of 3L3P sites as a limitation.